# Build an ML Feature Store with Iceberg

Machine learning models need stable, versioned features that can evolve safely. This template shows you how Iceberg's schema evolution and time travel capabilities work together to enable production-grade feature stores—ensuring your models train on consistent, auditable data while allowing safe feature experimentation.

## What You'll Learn

In under 10 minutes, you'll see how to:
- Build versioned ML features using PySpark on Snowflake (via Snowpark Connect)
- Use Iceberg's schema evolution to add new features without breaking production
- Query historical feature versions with time travel for reproducible training
- Compare feature distributions across versions for drift detection

## Why This Matters

Feature stores are critical infrastructure for ML teams, but they face a key challenge: features need to evolve (new signals discovered post-deployment) while maintaining stability (production models depend on consistent schemas). Iceberg solves this by providing backward-compatible versioning with zero-copy schema changes—letting you innovate on features while keeping production safe.


## Setup: Initialize Environment

First, let's set up our Snowflake environment and initialize a Spark session with Snowpark Connect.


In [ ]:
# Set up Snowflake environment
from snowflake.snowpark import Session

# Configure environment - using standard Snowflake Learning resources
session = Session.builder.getOrCreate()

session.sql("USE ROLE SNOWFLAKE_LEARNING_ROLE").collect()
session.sql("USE WAREHOUSE SNOWFLAKE_LEARNING_WH").collect()
session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()

# Create a unique schema for this template
schema_name = f"{session.sql('SELECT CURRENT_USER()').collect()[0][0]}_ICEBERG_FEATURE_STORE"
session.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}").collect()
session.sql(f"USE SCHEMA {schema_name}").collect()

print(f"✅ Environment configured successfully!")
print(f"📁 Using schema: {schema_name}")


In [ ]:
# Initialize Spark session with Snowpark Connect
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Create Spark session configured for Snowflake
spark = SparkSession.builder \
    .appName("Iceberg-Feature-Store") \
    .getOrCreate()

print("✅ Spark session initialized successfully!")
print(f"🎯 Spark version: {spark.version}")
print(f"📊 Ready to build versioned ML features with Iceberg!")


In [ ]:
# Load Tasty Bytes sample data
# Create and load the order_header table from S3
session.sql("""
CREATE OR REPLACE STAGE blob_stage
    URL = 's3://sfquickstarts/tastybytes/'
    FILE_FORMAT = (TYPE = CSV)
""").collect()

# Create the order_header table
session.sql("""
CREATE OR REPLACE TABLE order_header (
    order_id NUMBER(38,0),
    truck_id NUMBER(38,0),
    location_id NUMBER(38,0),
    customer_id NUMBER(38,0),
    discount_id NUMBER(38,0),
    shift_id NUMBER(38,0),
    shift_start_time TIME,
    shift_end_time TIME,
    order_channel VARCHAR(16777216),
    order_ts TIMESTAMP_NTZ,
    served_ts TIMESTAMP_NTZ,
    order_currency VARCHAR(3),
    order_amount NUMBER(38,4),
    order_tax_amount NUMBER(38,4),
    order_discount_amount NUMBER(38,4),
    order_total NUMBER(38,4)
)
""").collect()

# Load data from S3 stage
session.sql("""
COPY INTO order_header
FROM @blob_stage/raw_pos/order_header/
""").collect()

# Verify data loaded successfully
result = session.sql("SELECT COUNT(*) as order_count FROM order_header").collect()
print(f"✅ Loaded {result[0]['ORDER_COUNT']:,} orders from Tasty Bytes dataset")


## Version 1: Create Baseline RFM Features

RFM (Recency, Frequency, Monetary) analysis is a foundational technique in customer analytics. Let's create our initial feature set using PySpark to compute these metrics from customer order history.


In [ ]:
# Read order data using PySpark
orders_df = spark.table("order_header")

# Calculate RFM features using window functions
# Define the reference date for recency calculation
current_date_val = orders_df.agg(max("order_ts")).collect()[0][0]

# Compute RFM metrics per customer
rfm_features = orders_df.groupBy("customer_id").agg(
    # Recency: Days since last order
    datediff(lit(current_date_val), max("order_ts")).alias("recency_days"),
    
    # Frequency: Total number of orders
    count("order_id").alias("order_frequency"),
    
    # Monetary: Total revenue per customer
    sum("order_total").alias("monetary_value")
)

# Add feature_date column for partitioning
rfm_features = rfm_features.withColumn(
    "feature_date", 
    to_date(lit(current_date_val))
)

# Show sample of computed features
print("🎯 Sample RFM Features (Version 1):")
rfm_features.show(10)

print(f"\n✅ Computed RFM features for {rfm_features.count():,} customers")


In [ ]:
# Write Version 1 features to Iceberg table with partitioning
# Iceberg tables are created automatically when using USING ICEBERG clause
rfm_features.write \
    .mode("overwrite") \
    .option("iceberg.enabled", "true") \
    .partitionBy("feature_date") \
    .saveAsTable("CUSTOMER_FEATURES")

print("✅ Version 1 features saved to Iceberg table: CUSTOMER_FEATURES")
print("📊 Table is partitioned by feature_date for efficient querying")


In [ ]:
# Capture Version 1 snapshot ID for later time travel
# Query the Iceberg metadata to get the current snapshot
snapshot_df = session.sql("""
SELECT snapshot_id, committed_at, operation
FROM TABLE(INFORMATION_SCHEMA.TABLE_STORAGE_METRICS(TABLE_NAME => 'CUSTOMER_FEATURES'))
ORDER BY committed_at DESC
LIMIT 1
""").collect()

v1_snapshot_id = snapshot_df[0]['SNAPSHOT_ID']
v1_timestamp = snapshot_df[0]['COMMITTED_AT']

print(f"📸 Version 1 Snapshot Captured:")
print(f"   Snapshot ID: {v1_snapshot_id}")
print(f"   Timestamp: {v1_timestamp}")
print(f"   Schema: customer_id, recency_days, order_frequency, monetary_value, feature_date")
print(f"\n✅ This snapshot preserves the exact state of our baseline feature set")


## Version 2: Evolve Schema with New Feature

Now let's add a new feature to support lifetime value (LTV) prediction. This is a common scenario—you've deployed your feature store and now want to add new signals without breaking production pipelines.

Iceberg's schema evolution lets us add columns without rewriting existing data or pausing downstream consumers. Both versions remain accessible through time travel.


In [ ]:
# Read existing features from Iceberg table
existing_features = spark.table("CUSTOMER_FEATURES")

# Compute new LTV score feature
# LTV score combines frequency and monetary value with a multiplier
enriched_features = existing_features.withColumn(
    "customer_ltv_score",
    (col("monetary_value") * col("order_frequency") * lit(1.5)).cast("double")
)

print("🎯 Enhanced features with LTV score:")
enriched_features.select(
    "customer_id", 
    "recency_days", 
    "order_frequency", 
    "monetary_value", 
    "customer_ltv_score"
).show(10)

print("\n✅ New feature computed: customer_ltv_score")
print("📊 This metric helps predict long-term customer value")


In [ ]:
# Write with schema evolution enabled
# The mergeSchema option allows Iceberg to add new columns without rewriting existing data
enriched_features.write \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("CUSTOMER_FEATURES")

print("✅ Version 2 written with schema evolution!")
print("🔄 Schema evolved: added 'customer_ltv_score' column")
print("💾 Zero-copy operation: existing data not rewritten")
print("🔒 Production safe: v1 readers still see original 3-column schema")


In [ ]:
# Capture Version 2 snapshot ID
snapshot_df_v2 = session.sql("""
SELECT snapshot_id, committed_at, operation
FROM TABLE(INFORMATION_SCHEMA.TABLE_STORAGE_METRICS(TABLE_NAME => 'CUSTOMER_FEATURES'))
ORDER BY committed_at DESC
LIMIT 1
""").collect()

v2_snapshot_id = snapshot_df_v2[0]['SNAPSHOT_ID']
v2_timestamp = snapshot_df_v2[0]['COMMITTED_AT']

print(f"📸 Version 2 Snapshot Captured:")
print(f"   Snapshot ID: {v2_snapshot_id}")
print(f"   Timestamp: {v2_timestamp}")
print(f"   Schema: customer_id, recency_days, order_frequency, monetary_value, customer_ltv_score, feature_date")
print(f"\n✅ Both versions now coexist in the same table!")


## Compare Versions with Time Travel

This is where Iceberg's time travel capability shines. We can query both versions simultaneously—seeing exactly what features existed at any point in time. This is critical for ML reproducibility and debugging.


In [ ]:
# Query Version 1 using time travel
v1_features = spark.read \
    .option("snapshot_id", str(v1_snapshot_id)) \
    .table("CUSTOMER_FEATURES")

print("📸 Version 1 Features (3 columns):")
v1_features.select("customer_id", "recency_days", "order_frequency", "monetary_value").show(5)
print(f"Column count: {len(v1_features.columns)}")
print(f"Columns: {', '.join(v1_features.columns)}\n")

# Query Version 2 (current state)
v2_features = spark.table("CUSTOMER_FEATURES")

print("📸 Version 2 Features (4 columns):")
v2_features.select(
    "customer_id", 
    "recency_days", 
    "order_frequency", 
    "monetary_value", 
    "customer_ltv_score"
).show(5)
print(f"Column count: {len(v2_features.columns)}")
print(f"Columns: {', '.join(v2_features.columns)}\n")

print("✅ Both versions accessible simultaneously via time travel!")


In [ ]:
# Compare summary statistics across versions
print("📊 Version 1 Summary Statistics:")
v1_stats = v1_features.select(
    "recency_days", 
    "order_frequency", 
    "monetary_value"
).describe()
v1_stats.show()

print("\n📊 Version 2 Summary Statistics:")
v2_stats = v2_features.select(
    "recency_days", 
    "order_frequency", 
    "monetary_value",
    "customer_ltv_score"
).describe()
v2_stats.show()

print("✅ Base features (RFM) remain identical across versions")
print("🆕 New feature (LTV score) only in Version 2")
print("\n💡 This proves data integrity: schema evolution didn't corrupt existing features")


In [ ]:
# View complete snapshot history
print("📜 Complete Snapshot History:")
history_df = session.sql("""
SELECT 
    snapshot_id,
    committed_at,
    operation,
    summary
FROM TABLE(INFORMATION_SCHEMA.TABLE_STORAGE_METRICS(TABLE_NAME => 'CUSTOMER_FEATURES'))
ORDER BY committed_at ASC
""")

history_df.show(truncate=False)

print(f"\n✅ Snapshot History shows the evolution:")
print(f"   • {v1_timestamp}: Version 1 created (3 feature columns)")
print(f"   • {v2_timestamp}: Version 2 evolved (4 feature columns)")
print(f"\n🔄 Any snapshot can be queried for reproducible ML training")


### Real-World Use Cases

Here's what this capability enables for production ML systems:

**🔬 A/B Testing Features**
- Control group uses Version 1 (3 features)
- Treatment group uses Version 2 (4 features)
- Compare model performance without data duplication

**🐛 Model Debugging**
- Production model degrading? Query the exact snapshot used during training
- Compare feature distributions between training and inference snapshots
- Identify data drift or schema mismatches instantly

**📈 Safe Feature Rollout**
- Deploy new features (v2) to production without migration downtime
- Existing models continue using v1 snapshots
- Rollback instantly if issues arise—just point to previous snapshot

**✅ Compliance & Auditing**
- "What features were available when we made this prediction?"
- Complete lineage of schema evolution for regulated industries
- Reproduce exact training conditions for audit purposes


## Key Takeaways

You've just built a production-ready ML feature store with versioning capabilities. Here's what you learned:

### Core Capabilities

1. **Snowpark Connect + PySpark** → Use familiar DataFrame API on Snowflake infrastructure without managing Spark clusters

2. **Iceberg Schema Evolution** → Add features incrementally without rewriting data or breaking production pipelines

3. **Time Travel for Reproducibility** → Query exact feature snapshots used during training, critical for debugging and compliance

4. **Zero-Copy Versioning** → Schema changes create new versions instantly without data duplication or downtime

### Why This Matters for ML

Traditional feature stores force a choice: **stability or agility**. You either lock schemas (blocking innovation) or accept breaking changes (risking production).

Iceberg solves this by providing **backward-compatible versioning**:
- Production models stay stable (query v1 snapshot)
- New experiments use latest features (query v2 snapshot)
- Both coexist in the same table with full ACID guarantees

This architecture enables continuous feature development without production risk—exactly what modern ML teams need.


## Cleanup

Let's clean up the resources we created during this template.


In [ ]:
# Drop the Iceberg table
session.sql("DROP TABLE IF EXISTS CUSTOMER_FEATURES").collect()

# Drop the source table
session.sql("DROP TABLE IF EXISTS order_header").collect()

# Drop the stage
session.sql("DROP STAGE IF EXISTS blob_stage").collect()

# Clear Spark cache
spark.catalog.clearCache()

print("✅ Cleanup complete!")
print("🎯 All tables and stages removed")
print("💾 Spark cache cleared")
print("\n🎉 You've successfully built an ML feature store with Iceberg!")
print("📚 Ready to apply these patterns to your own feature engineering workflows.")


## Additional Resources

### Documentation
- [Snowflake Iceberg Tables Documentation](https://docs.snowflake.com/en/user-guide/tables-iceberg)
- [Snowpark Connect for Apache Spark](https://docs.snowflake.com/en/developer-guide/snowpark-connect/snowpark-connect-overview)
- [Apache Iceberg Table Format](https://iceberg.apache.org/)

### Next Steps
- Explore advanced Iceberg features like partition evolution and metadata tables
- Build real-time feature pipelines with Snowpipe and Iceberg
- Integrate with ML frameworks (scikit-learn, XGBoost, TensorFlow)
- Implement feature serving patterns for low-latency inference

### Related Topics
- Time travel queries for temporal analysis
- Schema evolution patterns for data warehousing
- Feature store best practices for ML engineering
- Open table formats and data lakehouse architectures
